In [ ]:
def operator(name, args):
    x, y = args[0], args[-1]
    if name == "PICKUP":
        return ([("CLEAR", x), ("ONTABLE", x), ("HANDEMPTY",)],
                [("HOLDING", x)],
                [("CLEAR", x), ("ONTABLE", x), ("HANDEMPTY",)])
    if name == "PUTDOWN":
        return ([("HOLDING", x)],
                [("ONTABLE", x), ("CLEAR", x), ("HANDEMPTY",)],
                [("HOLDING", x)])
    if name == "STACK":
        return ([("HOLDING", x), ("CLEAR", y)],
                [("ON", x, y), ("CLEAR", x), ("HANDEMPTY",)],
                [("HOLDING", x), ("CLEAR", y)])
    return ([("ON", x, y), ("CLEAR", x), ("HANDEMPTY",)],
            [("HOLDING", x), ("CLEAR", y)],
            [("ON", x, y), ("CLEAR", x), ("HANDEMPTY",)])


def achiever(pred, state):
    kind, x = pred[0], pred[1]
    if kind == "ON":
        return "STACK", (x, pred[2])
    if kind == "ONTABLE":
        return "PUTDOWN", (x,)
    if kind == "CLEAR":
        on = next((p for p in state if p[0] == "ON" and p[2] == x), None)
        return "UNSTACK", (on[1], x) if on else None
    if kind == "HOLDING":
        if ("ONTABLE", x) in state:
            return "PICKUP", (x,)
        on = next(p for p in state if p[0] == "ON" and p[1] == x)
        return "UNSTACK", (x, on[2])
    held = next((p for p in state if p[0] == "HOLDING"), None)
    return "PUTDOWN", (held[1],) if held else None


def goal_stack_plan(initial, goal):
    state, stack, plan = set(initial), [("MULTI", goal)], []
    while stack:
        top = stack.pop()
        if top[0] == "MULTI":
            if all(p in state for p in top[1]):
                continue
            stack.append(top)
            stack += [("PRED", p) for p in reversed(top[1]) if p not in state]
        elif top[0] == "PRED":
            if top[1] in state:
                continue
            name, args = achiever(top[1], state)
            stack.append(("ACTION", name, args))
            stack.append(("MULTI", operator(name, args)[0]))
        else:
            _, name, args = top
            _, add, delete = operator(name, args)
            state -= set(delete)
            state |= set(add)
            plan.append((name, args))
    return plan, state


initial = [
    ("ONTABLE", "A"), ("ONTABLE", "B"), ("ONTABLE", "C"),
    ("CLEAR", "A"), ("CLEAR", "B"), ("CLEAR", "C"), ("HANDEMPTY",),
]
goal = [("ON", "B", "C"), ("ON", "A", "B")]  # B on C first, then A on B

plan, final_state = goal_stack_plan(initial, goal)

print("Initial state:", sorted(initial))
print("Goal:", goal, "\n")
for i, (name, args) in enumerate(plan, 1):
    print("Step", i, ":", name, *args)
print("\nGoal achieved:", all(g in final_state for g in goal))








































# ===== THEORY (copy to journal, then delete this block) =====
# Practical B6 : Goal Stack Planning - Blocks World
#
# OBJECTIVE:
#   Solve the Blocks World problem using Goal Stack Planning and achieve a specified goal configuration.
#
# THEORY:
#   Goal stack planning works backward from the goal using a stack of goals, predicates and actions.
#   The operators PICKUP, PUTDOWN, STACK and UNSTACK have preconditions, add lists and delete lists.
#   The achiever selects an operator that can achieve a predicate, and its preconditions are pushed onto the stack.
#   When an action reaches the top of the stack, its add and delete lists are applied to the state.
#
# ALGORITHM:
#   1. Define the four operators with their preconditions, add lists and delete lists.
#   2. Initialise the state, the goal stack with the goal and an empty plan.
#   3. Pop the top item of the stack.
#   4. If it is a multi-goal, push every goal that is not yet satisfied.
#   5. If it is a predicate, choose an operator with the achiever and push the action and its preconditions.
#   6. If it is an action, apply its delete and add lists to the state and append it to the plan.
#   7. Repeat until the stack is empty.
#   8. Print the plan and check whether the goal is achieved.
#
# CONCLUSION:
#   Goal stack planning produces a valid action sequence that reaches the goal.
#   Choosing the goal order carefully avoids unnecessary work and gives a short plan.
# ===== END THEORY =====
